In [1]:
"""

Hourly Data Assimilation and Spatial Interpolation

Part A: Build an hourly index covering the study period 
1. Station data
    1a. Joins station data csvs with the metadata csv to bring in elevation, lat/long associated with each station id 
    1b. Collapse station data to the hourly level by... at each target hour, collect all station observations within the hour and 
    average for that station
    1c. Generally the station csvs contain predictors for temp_air, temp_dew, and rh. For any predictors that were missing 
    before (i.e., NA), calculate them using foundational equations found in model_meteo(). 
    Calculate temp_bulb based on equations found in model_meteo(). 
2. IMERG: 
    2a. Convert wide to long and average half hourly data to the hourly level. 
3. MRoS: 
    3a. There might be multiple observations coming from the same observer within an hour. 
    If that's the case, choose the latter observation that was recorded (i.e., if an observer changed their mind about the phase). 
    Otherwise, floor each MRoS observation datetime_UTC to the starting hour. 
4. At this point, all the data should have lat, lon, datetime_utc (hourly level), predictors. 
    Filter all of them to the lat/long within our DEM AOI. 

Part B: IDW to Surface
Now that all data should be time synchronized at the hourly level, perform spatial interpolations onto the 10 m DEM grid. 
1. Resample the DEM surface to be 1km to free up some compute time down the road. Reproject from degrees to meters.
2. IDW to grid: perform a simple IDW on each predictor to the DEM surface/grid. 
    The predictors we use are 
    a) PLP from the imerg dataset,0
    b)  mros_plp_proxy from the MRoS dataset (rain --> 100, snow --> 0, mix --> 50 % prob to match IMERG PLP format), 
    c) t_air, t_wet, t_dew, rh from station datasets (apply lapse rate -0.0005 K m-1 to these variables, except for RH, which is dimensionless)
    Use projected coordinates, KDTree for N-nearest, IDW power, and require minimum of 3 points

"""

# Dependencies: pandas, numpy, pyarrow, geopandas, shapely, rasterio, rioxarray, xarray,
#               pyproj, scipy (KDTree), tqdm

import re
import json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import rasterio as rio
from rasterio.warp import transform_bounds, reproject, Resampling, calculate_default_transform
import xarray as xr
import rioxarray  # noqa: F401 (registers .rio accessor)
from pyproj import CRS, Transformer
from scipy.spatial import cKDTree
from tqdm import tqdm
import pytz
from collections import defaultdict
import math
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import netCDF4


In [2]:
# --------------------------- CONFIG ---------------------------------
BASE_DIR = Path().resolve().parent  # current working dir
print("BASE_DIR:", BASE_DIR)


CONFIG = {
    "wy_start": "2024-10-01T00:00:00Z",
    "wy_end":   "2025-05-31T23:59:59Z",
    # "test_start": "2025-02-01T00:00:00Z",   # narrow test window first
    # "test_end":   "2025-04-01T00:00:00Z",
    "test_start": "2024-10-01T00:00:00Z",   # Entire window
    "test_end":   "2025-05-31T23:59:59Z",

    "station_meta_csv": BASE_DIR / "Data/Stations/station_metadata_20241001_20250531.csv",
    "station_dir": BASE_DIR / "Data/Stations",   # per-station CSVs
    "imerg_dir":   BASE_DIR / "Data/IMERG",      # parquet (wide)
    "mros_parquet": BASE_DIR / "Data/observations/wy25_mros_obs.parquet",

    "dem_path": "C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/DEM_AOI_TNM_10m.tif",
    "out_dir":  BASE_DIR / "outputs/hourly_pipeline",

    "idw_power": 2.0,
    "k_nearest": 8,
    "min_points": 3,
    "lapse_K_per_m": -0.005,   # constant lapse for temps
    "proj_fallback": "EPSG:3310"  # if DEM is geographic
}

out_dir = Path(CONFIG["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

BASE_DIR: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype


In [3]:
# ------------------------- UTIL: time --------------------------------
def to_utc(dt_series: pd.Series) -> pd.DatetimeIndex:
    """Force timestamps to UTC, making naive → UTC-naive assumed in UTC."""
    dt = pd.to_datetime(dt_series, errors="coerce", utc=True)
    # If dt_series had naive datetimes and pandas assumed local, .tz_convert('UTC') not needed.
    return dt

def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
                         freq="H", tz="UTC")

def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

In [4]:
# ---------------------- UTIL: meteorology functions -------------------------
# Magnus (Tetens) saturation vapor pressure over water (°C)
def esat_hpa(Tc: float) -> float:
    return 6.112 * np.exp(17.67 * Tc / (Tc + 243.5))

def td_from_ta_rh(TaC: np.ndarray, RH: np.ndarray) -> np.ndarray:
    """Dewpoint from air temp (°C) and RH (%)"""
    TaC = np.asarray(TaC, dtype=float)
    RH = np.clip(np.asarray(RH, dtype=float), 1e-6, 100.0)
    a, b = 17.625, 243.04
    gamma = np.log(RH / 100.0) + (a * TaC) / (b + TaC)
    Td = (b * gamma) / (a - gamma)
    return Td

def rh_from_ta_td(TaC: np.ndarray, TdC: np.ndarray) -> np.ndarray:
    """RH (%) from air temp and dewpoint (°C)"""
    TaC = np.asarray(TaC, dtype=float)
    TdC = np.asarray(TdC, dtype=float)
    a, b = 17.625, 243.04
    ln_es_Ta = (a * TaC) / (b + TaC)
    ln_es_Td = (a * TdC) / (b + TdC)
    RH = 100.0 * np.exp(ln_es_Td - ln_es_Ta)
    return np.clip(RH, 0.0, 100.0)

def tw_stull(TaC: np.ndarray, RH: np.ndarray) -> np.ndarray:
    """
    Wet-bulb approximation (°C) by Stull (2011).
    TaC in °C, RH in %
    """
    TaC = np.asarray(TaC, dtype=float)
    RH = np.clip(np.asarray(RH, dtype=float), 1e-6, 100.0)
    Tw = (TaC * np.arctan(0.151977 * np.sqrt(RH + 8.313659)) +
          np.arctan(TaC + RH) - np.arctan(RH - 1.676331) +
          0.00391838 * RH**1.5 * np.arctan(0.023101 * RH) - 4.686035)
    return Tw

def fill_station_row_vars(df: pd.DataFrame) -> pd.DataFrame:
    """
    Given columns temp_air, temp_dew, rh (percent), compute missing among them,
    then compute temp_wet (Tw) where possible.
    """
    Ta = df.get("temp_air")
    Td = df.get("temp_dew")
    RH = df.get("rh")

    # Any two → compute third
    if "temp_air" in df and "temp_dew" in df and "rh" not in df:
        df["rh"] = rh_from_ta_td(Ta, Td)
    if "temp_air" in df and "rh" in df and "temp_dew" not in df:
        df["temp_dew"] = td_from_ta_rh(Ta, RH)
    if "temp_dew" in df and "rh" in df and "temp_air" not in df:
        pass

    # Clamp RH
    if "rh" in df:
        df["rh"] = np.clip(df["rh"].astype(float), 0.0, 100.0)

    # Wet-bulb
    if "temp_air" in df and "rh" in df:
        df["temp_wet"] = tw_stull(df["temp_air"].astype(float), df["rh"].astype(float))

    return df


In [5]:
# # -------------------- LOAD: DEM & AOI functions ------------------------

def load_dem_and_aoi(dem_path: str):
    with rio.open(dem_path) as src:
        dem_crs = CRS.from_wkt(src.crs.to_wkt()) if src.crs else None
        bounds = src.bounds
        aoi_wgs84 = transform_bounds(src.crs, "EPSG:4326",
                                     bounds.left, bounds.bottom, bounds.right, bounds.top,
                                     densify_pts=21)
    aoi_poly = box(aoi_wgs84[0], aoi_wgs84[1], aoi_wgs84[2], aoi_wgs84[3])
    return dem_path, dem_crs, aoi_poly

# def reproject_resample_dem_to_1km(dem_path: str, proj_fallback="EPSG:3310"):
#     with rio.open(dem_path) as src:
#         src_crs = CRS.from_wkt(src.crs.to_wkt()) if src.crs else None
#         if not src_crs or src_crs.is_geographic:
#             dst_crs = CRS.from_string(proj_fallback)
#             transform, width, height = calculate_default_transform(
#                 src.crs, dst_crs, src.width, src.height, *src.bounds
#             )
#             profile = src.profile.copy()
#             profile.update(crs=dst_crs, transform=transform, width=width, height=height)
#             data_proj = np.empty((height, width), dtype="float32")
#             reproject(
#                 source=rio.band(src, 1),
#                 destination=data_proj,
#                 src_transform=src.transform,
#                 src_crs=src.crs,
#                 dst_transform=transform,
#                 dst_crs=dst_crs,
#                 resampling=Resampling.bilinear
#             )
#             dem_proj, profile = data_proj, profile
#         else:
#             dem_proj = src.read(1).astype("float32")
#             profile = src.profile.copy()

#     # Resample to ~1 km via average
#     xres = abs(profile["transform"].a)
#     yres = abs(profile["transform"].e)
#     target_res = 1000.0
#     new_w = max(1, int(np.floor(profile["width"]  * (xres/target_res))))
#     new_h = max(1, int(np.floor(profile["height"] * (abs(yres)/target_res))))

#     dst = np.empty((new_h, new_w), dtype=np.float32)
#     dst_transform = rio.Affine(
#         target_res, 0.0, profile["transform"].c,
#         0.0, -target_res, profile["transform"].f
#     )
#     reproject(
#         source=dem_proj,
#         destination=dst,
#         src_transform=profile["transform"],
#         src_crs=profile["crs"],
#         dst_transform=dst_transform,
#         dst_crs=profile["crs"],
#         resampling=Resampling.average
#     )
#     out_prof = profile.copy()
#     out_prof.update({"height": new_h, "width": new_w, "transform": dst_transform,
#                      "dtype":"float32", "count":1})
#     return dst, out_prof

def grid_centers(profile):
    T = profile["transform"]
    xs = T.c + (np.arange(profile["width"]) + 0.5) * T.a
    ys = T.f + (np.arange(profile["height"]) + 0.5) * T.e
    X, Y = np.meshgrid(xs, ys)
    return np.column_stack([X.ravel(), Y.ravel()])

# Load DEM/AOI now
dem_path, dem_crs, aoi_poly = load_dem_and_aoi(CONFIG["dem_path"])
# dem1k_data, dem1k_profile = reproject_resample_dem_to_1km(dem_path, CONFIG["proj_fallback"])
# grid_xy = grid_centers(dem1k_profile)
# grid_elev = dem1k_data.ravel()
# proj_crs = dem1k_profile["crs"]

# print(f"DEM 1-km grid: {dem1k_profile['width']} x {dem1k_profile['height']} | "
#       f"res ≈ {abs(dem1k_profile['transform'].a)} m")


In [6]:
# # Save resampled DEM to GeoTIFF
# out_dem_path = out_dir / "DEM_1km.tif"

# with rio.open(out_dem_path, "w", **dem1k_profile) as dst:
#     dst.write(dem1k_data, 1)

# print(f"Saved 1-km DEM to {out_dem_path}")


In [7]:
# Load already projected and saved 1km DEM tif

with rio.open(r"C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/DEM_1km.tif") as src:
    dem1k_profile = src.profile   # metadata
    dem1k_data = src.read(1)      # pixel values

    # Optional extras
    dem_crs = src.crs             # CRS object
    dem_bounds = src.bounds       # bounding box
    dem_transform = src.transform # affine transform

grid_xy = grid_centers(dem1k_profile)
grid_elev = dem1k_data.ravel()
proj_crs = dem1k_profile["crs"]

print(f"DEM 1-km grid: {dem1k_profile['width']} x {dem1k_profile['height']} | "
      f"res ≈ {abs(dem1k_profile['transform'].a)} m")


DEM 1-km grid: 271 x 501 | res ≈ 1000.0 m


In [8]:
# -------------------- LOAD: Stations ---------------------------------

def load_station_meta(meta_csv: str) -> pd.DataFrame:
    meta = pd.read_csv(meta_csv)
    req = {"id","lat","lon","elev","timezone_lst"}
    missing = req - set(meta.columns)
    if missing:
        raise ValueError(f"Station metadata missing columns: {missing}")
    meta["id"] = meta["id"].astype(str)
    return meta

def load_station_timeseries(station_dir: str, meta: pd.DataFrame) -> pd.DataFrame:
    files = [p for p in Path(station_dir).glob("*.csv") if "meta" not in p.name.lower()]
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        if "id" not in df.columns:
            df["id"] = f.stem
        keep = ["id","datetime","temp_air","temp_dew","rh"]
        for k in keep:
            if k not in df.columns:
                df[k] = np.nan
        df = df[keep]
        df["id"] = df["id"].astype(str)

        # timezone per station
        tz_vals = meta.loc[meta["id"] == df["id"].iloc[0], "timezone_lst"].values
        dt_local = pd.to_datetime(df["datetime"], errors="coerce")
        if len(tz_vals) == 1:
            try:
                tz = pytz.timezone(tz_vals[0])
                if getattr(dt_local.dt, "tz", None) is None:
                    df["datetime"] = dt_local.dt.tz_localize(tz, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
                else:
                    df["datetime"] = dt_local.dt.tz_convert("UTC")
            except Exception as e:
                print(f"Warning: timezone '{tz_vals}' failed for station {df['id'].iloc[0]}: {e}")
                df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
        else:
            df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
        dfs.append(df)
    if not dfs:
        return pd.DataFrame(columns=["id","datetime","temp_air","temp_dew","rh"])
    return pd.concat(dfs, ignore_index=True)

def hourly_station_agg(st_df: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    df = st_df.merge(meta, on="id", how="left")
    df["hour_utc"] = df["datetime"].dt.floor("H")
    agg = (df.groupby(["id","hour_utc"], as_index=False)
             .agg(temp_air=("temp_air","mean"),
                  temp_dew=("temp_dew","mean"),
                  rh=("rh","mean"),
                  lat=("lat","first"),
                  lon=("lon","first"),
                  elev=("elev","first")))
    agg = fill_station_row_vars(agg)
    return agg

# run
meta = load_station_meta(CONFIG["station_meta_csv"])
st_ts = load_station_timeseries(CONFIG["station_dir"], meta)
st_hr = hourly_station_agg(st_ts, meta)
# window filter
st_hr = st_hr[(st_hr["hour_utc"] >= pd.to_datetime(CONFIG["test_start"])) &
              (st_hr["hour_utc"] <= pd.to_datetime(CONFIG["test_end"]))]

print(f"Stations hourly rows in window: {len(st_hr)}")


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_27180\1927537202.py:48: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour_utc"] = df["datetime"].dt.floor("H")


Stations hourly rows in window: 692162


In [9]:
# -------------------- LOAD: IMERG (wide→long→hourly) -----------------

# Dates like "2024-10-11 01:30:00" (space or "T" also OK)
TIME_COL_RE = re.compile(r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}(?::\d{2})?$")
DATE_ONLY_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def read_imerg_wide_to_long(path: Path) -> pd.DataFrame:
    import pyarrow.parquet as pq
    table = pq.read_table(path)
    df = table.to_pandas()
    df = df.rename(columns={"x": "lon", "y": "lat"})

    # Only use the columns that include a time; drop pure date columns
    time_cols = [c for c in df.columns if TIME_COL_RE.match(str(c))]
    if not time_cols:
        # fall back: if nothing matched, try any time-like and then remove date-only
        any_timeish = [c for c in df.columns if re.match(r"^\d{4}-\d{2}-\d{2}", str(c))]
        time_cols = [c for c in any_timeish if not DATE_ONLY_RE.match(str(c))]
    if not time_cols:
        raise ValueError(f"No time-like (HH:MM) columns in {path}")

    long = df.melt(
        id_vars=["lat", "lon"],
        value_vars=time_cols,
        var_name="time_str",
        value_name="plp_raw",
    )

    # Parse timestamps as UTC; these already include hour/minute
    long["time_utc"] = pd.to_datetime(long["time_str"], utc=True, errors="coerce")

    # Values → float; scale to percent if the data are 0–1
    plp = pd.to_numeric(long["plp_raw"], errors="coerce").astype(float)
    if np.nanmax(plp) <= 1.0:
        plp *= 100.0
    long["plp"] = plp

    return long.loc[long["time_utc"].notna(), ["time_utc", "lat", "lon", "plp"]]

def hourly_imerg(imerg_dir: str, start_iso: str, end_iso: str) -> pd.DataFrame:
    files = list(Path(imerg_dir).rglob("*.parquet"))
    if not files:
        print(f"[IMERG] No parquet files under {imerg_dir}")
        return pd.DataFrame(columns=["hour_utc", "lat", "lon", "plp"])

    start_ts = pd.to_datetime(start_iso, utc=True)
    end_ts   = pd.to_datetime(end_iso,   utc=True)

    dfs = []
    for f in files:
        df = read_imerg_wide_to_long(f)
        df = df[(df["time_utc"] >= start_ts) & (df["time_utc"] <= end_ts)]
        if not df.empty:
            dfs.append(df)

    if not dfs:
        print("[IMERG] Found files but no rows within the requested window.")
        return pd.DataFrame(columns=["hour_utc", "lat", "lon", "plp"])

    imerg = pd.concat(dfs, ignore_index=True)

    # Average the two half-hour slots within each hour per pixel
    imerg["hour_utc"] = imerg["time_utc"].dt.floor("h")
    imerg_hr = (imerg.groupby(["hour_utc", "lat", "lon"], as_index=False)
                      .agg(plp=("plp", "mean")))
    return imerg_hr

# run
imerg_hr = hourly_imerg(CONFIG["imerg_dir"], CONFIG["test_start"], CONFIG["test_end"])
print(f"IMERG hourly points in window: {len(imerg_hr)}")


IMERG hourly points in window: 7358160


In [10]:
# -------------------- LOAD: MRoS -------------------------------------
def load_mros(mros_parquet: str, start_iso: str, end_iso: str) -> pd.DataFrame:
    import pyarrow.parquet as pq
    table = pq.read_table(mros_parquet)
    df = table.to_pandas()

    if "datetime_utc" not in df.columns:
        dt = pd.to_datetime(df["date_submitted_utc"] + " " + df["time_submitted_utc"],
                            utc=True, errors="coerce")
        df["datetime_utc"] = dt
    df["hour_utc"] = df["datetime_utc"].dt.floor("H")
    df["phase"] = df["phase"].str.lower()

    key_cols = ["hour_utc"]
    if "observer_id" in df.columns:
        key_cols.append("observer_id")
    else:
        df["lat_bin"] = pd.to_numeric(df["latitude"], errors="coerce").round(4)
        df["lon_bin"] = pd.to_numeric(df["longitude"], errors="coerce").round(4)
        key_cols += ["lat_bin","lon_bin"]

    df = df.sort_values("datetime_utc")
    last = df.groupby(key_cols, as_index=False).tail(1)

    map_plp = {"snow":0.0, "mix":50.0, "rain":100.0}
    last["mros_plp_proxy"] = last["phase"].map(map_plp).astype(float)

    last = last[(last["hour_utc"] >= pd.to_datetime(start_iso)) &
                (last["hour_utc"] <= pd.to_datetime(end_iso))]
    return last.rename(columns={"latitude":"lat","longitude":"lon"})[
        ["hour_utc","lat","lon","mros_plp_proxy","phase"]
    ]

# run
mros = load_mros(CONFIG["mros_parquet"], CONFIG["test_start"], CONFIG["test_end"])
print(f"MRoS hourly rows in window: {len(mros)}")



C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_27180\2458660098.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df["date_submitted_utc"] + " " + df["time_submitted_utc"],


MRoS hourly rows in window: 28678


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_27180\2458660098.py:11: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour_utc"] = df["datetime_utc"].dt.floor("H")
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_27180\2458660098.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last["mros_plp_proxy"] = last["phase"].map(map_plp).astype(float)


In [11]:
# # Load hourly-level stations, IMERG, and MRoS if already performed:

# st_hr   = pd.read_parquet(out_dir / "stations_hourly.parquet")
# imerg_hr = pd.read_parquet(out_dir / "imerg_hourly.parquet")
# mros     = pd.read_parquet(out_dir / "mros_hourly.parquet")


In [12]:
# -------------------- AOI filter -------------------------------------
def filter_points_to_aoi(df: pd.DataFrame, aoi_poly) -> pd.DataFrame:
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    poly = gpd.GeoSeries([aoi_poly], crs="EPSG:4326").iloc[0]
    mask = g.intersects(poly)
    return df.loc[mask.values].drop(columns=["geometry"], errors="ignore")

st_hr   = filter_points_to_aoi(st_hr,   aoi_poly)
imerg_hr= filter_points_to_aoi(imerg_hr,aoi_poly)
mros    = filter_points_to_aoi(mros,    aoi_poly)

print(len(st_hr), len(imerg_hr), len(mros))


557766 6966000 7896


In [13]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert DataFrame → Arrow Table → Parquet
pq.write_table(pa.Table.from_pandas(st_hr), out_dir / "stations_hourly.parquet")
pq.write_table(pa.Table.from_pandas(imerg_hr), out_dir / "imerg_hourly.parquet")
pq.write_table(pa.Table.from_pandas(mros), out_dir / "mros_hourly.parquet")


In [14]:
# How many MRoS reports per hour?
mros_counts = mros.groupby("hour_utc").size().rename("n_mros")
print(mros_counts.describe())
print("hours with >=3:", (mros_counts>=3).sum(), " | >=2:", (mros_counts>=2).sum(), " | >=1:", (mros_counts>=1).sum())

h = pd.Timestamp("2024-10-17 21:00:00+00:00")
print(mros.loc[mros["hour_utc"]==h, ["lat","lon","phase","mros_plp_proxy"]].head(10))


count    1067.000000
mean        7.400187
std         8.854469
min         1.000000
25%         1.000000
50%         4.000000
75%        10.000000
max        65.000000
Name: n_mros, dtype: float64
hours with >=3: 670  | >=2: 794  | >=1: 1067
                  lat                lon phase  mros_plp_proxy
114        39.5591129       -119.6870391   mix            50.0
74   39.5133430166667  -119.781537466667  rain           100.0
130        39.5600358        -119.744263  rain           100.0
80         39.8056058        -120.468989  snow             0.0
89   39.8175302585422  -120.315823129357  rain           100.0
135        39.6024208       -119.8693104  rain           100.0
126  39.0883375833333  -119.775830883333  rain           100.0


## Part B: IDW Interpolation

In [15]:
# -------------------- IDW Functions ------------------------------------
def build_transformer(src_epsg: str, dst_crs):
    return Transformer.from_crs(src_epsg, dst_crs, always_xy=True)

def idw_grid_from_points(hour_points: pd.DataFrame,
                         grid_xy: np.ndarray,
                         grid_elev: np.ndarray,
                         proj_crs,
                         idw_power=2.0, k=8, min_points=3,
                         value_col="temp_air",
                         station_elev_col="elev",
                         apply_lapse=False, lapse=-0.005):
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"])
    if pts.empty or pts[value_col].notna().sum() < min_points:
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    # transform station coords into the same projection
    tf = build_transformer("EPSG:4326", proj_crs)
    px, py = tf.transform(pts["lon"].values, pts["lat"].values)
    P = np.column_stack([px, py])

    values = pts[value_col].values.astype(float)
    stn_elev = pts[station_elev_col].values.astype(float) if station_elev_col in pts else np.zeros_like(values)

    # nearest neighbor search, for each grid cell, finds up to k nearest stations
    tree = cKDTree(P)
    dists, idxs = tree.query(grid_xy, k=min(k, len(P)))
    if dists.ndim == 1:
        dists = dists[:, None]
        idxs  = idxs[:,  None]

    # get neighbor station values for each grid cell, apply lapse rate on select parameters to account for temp change with elevation
    v_neighbors = values[idxs]
    if apply_lapse:
        zc = grid_elev[:, None]
        zj = stn_elev[idxs]
        v_neighbors = v_neighbors + lapse * (zc - zj) # if grid cell is higher than the station, reduce the interpolated temperature

    # compute weights
    with np.errstate(divide="ignore"):
        w = 1.0 / np.power(dists, idw_power)
    w[np.isinf(w)] = 1e12
    w[~np.isfinite(w)] = 0.0
    # normalize weightsm ensure weights sum to 1 per cell
    w_sum = w.sum(axis=1, keepdims=True)
    w_norm = np.divide(w, w_sum, out=np.zeros_like(w), where=w_sum > 0)

    valid_counts = np.sum(w > 0, axis=1)
    # weighted sum (weighted average of neighbor values)
    grid_vals = np.sum(w_norm * v_neighbors, axis=1)
    grid_vals[valid_counts < min_points] = np.nan
    return grid_vals.astype(np.float32)


In [16]:
# -------------------- Hourly Assimilation ------------------------------------

out_dir = Path(CONFIG["out_dir"]); out_dir.mkdir(parents=True, exist_ok=True)
hours = hourly_index(CONFIG["test_start"], CONFIG["test_end"])

variables = [
    ("temp_air",       "station", True), # true for using lapse rate during IDW
    ("temp_dew",       "station", True),
    ("temp_wet",       "station", True),
    ("rh",             "station", False), # false for not using lapse rate during IDW
    ("mros_plp_proxy", "mros",    False),
    ("plp",            "imerg",   False),
]

# Build coords from the DEM 1-km profile (use rasterio.xy to avoid any drift)
from rasterio.transform import xy as rio_xy
H, W = dem1k_profile["height"], dem1k_profile["width"]
T = dem1k_profile["transform"]

rows = np.arange(H)
cols = np.arange(W)
# centers from affine; one row vector for x, one col vector for y
x_centers = np.array([rio_xy(T, 0.5, c + 0.5, offset="center")[0] for c in cols])
y_centers = np.array([rio_xy(T, r + 0.5, 0.5, offset="center")[1] for r in rows])

coords = {
    "time": hours,
    "y": y_centers,
    "x": x_centers,
}
data_vars = {
    name: np.full((len(hours), H, W), np.nan, dtype=np.float32)
    for (name, _, _) in variables
}

def summarize_points(st_t, mros_t, imerg_t, min_points):
    msg = []
    ns = st_t.dropna(subset=["temp_air","temp_dew","rh"]).shape[0]
    msg.append(f"stations rows: {ns}")
    msg.append(f"mros rows: {mros_t.dropna(subset=['mros_plp_proxy']).shape[0]}")
    msg.append(f"imerg rows: {imerg_t.dropna(subset=['plp']).shape[0]}")
    msg.append("vars_ok: " + ", ".join([
        f"Ta={int(st_t['temp_air'].notna().sum()>=min_points)}",
        f"Td={int(st_t['temp_dew'].notna().sum()>=min_points)}",
        f"Tw={int(('temp_wet' in st_t) and (st_t['temp_wet'].notna().sum()>=min_points))}",
        f"RH={int(st_t['rh'].notna().sum()>=min_points)}",
        f"MRoS={int(mros_t['mros_plp_proxy'].notna().sum()>= 1 )}",
        f"PLP={int(imerg_t['plp'].notna().sum()>=min_points)}"
    ]))
    return " | ".join(msg)

for ti, t in enumerate(tqdm(hours, desc="Hourly surfaces", ncols=88)):
    st_t   = st_hr[st_hr["hour_utc"] == t]
    mros_t = mros[mros["hour_utc"] == t]
    imerg_t= imerg_hr[imerg_hr["hour_utc"] == t]

    print(f"[{print_time(t)}] {summarize_points(st_t, mros_t, imerg_t, CONFIG['min_points'])}")

    for name, src, use_lapse in tqdm(variables, desc=f"  vars {print_time(t)}", leave=False, ncols=88):
        if src == "station":
            npts = st_t["temp_air"].notna().sum()
            print(f"[{print_time(t)}] Station points available: {npts}")
            if st_t.empty or st_t[name].notna().sum() < CONFIG["min_points"]:
                continue
            pts = st_t[["lon","lat","elev", name]]
            vals = idw_grid_from_points(
                pts, grid_xy, grid_elev, proj_crs,
                idw_power=CONFIG["idw_power"], k=CONFIG["k_nearest"],
                min_points=CONFIG["min_points"], value_col=name,
                station_elev_col="elev",
                apply_lapse=use_lapse, lapse=CONFIG["lapse_K_per_m"]
            )
            data_vars[name][ti, :, :] = vals.reshape(H, W)

        elif src == "mros":
            npts = mros_t["mros_plp_proxy"].notna().sum()
            print(f"[{print_time(t)}] MRoS points available: {npts}")
            if npts < 1:   # require at least 1 obs
                continue
            pts = mros_t.rename(columns={"mros_plp_proxy":"val"})[["lon","lat","val"]].assign(elev=0.0)
            # allow looser threshold for MRoS
            vals = idw_grid_from_points(
                pts, grid_xy, grid_elev, proj_crs,
                idw_power=CONFIG["idw_power"], k=CONFIG["k_nearest"],
                min_points=1,
                value_col="val",
                station_elev_col="elev", apply_lapse=False
            )
            data_vars[name][ti, :, :] = vals.reshape(H, W)

            
        elif src == "imerg":
            npts = imerg_t["plp"].notna().sum()
            print(f"[{print_time(t)}] IMERG points available: {npts}")
            if imerg_t.empty or imerg_t["plp"].notna().sum() < CONFIG["min_points"]:
                continue
            pts = imerg_t.rename(columns={"plp":"val"})[["lon","lat","val"]].assign(elev=0.0)
            vals = idw_grid_from_points(
                pts, grid_xy, grid_elev, proj_crs,
                idw_power=CONFIG["idw_power"], k=CONFIG["k_nearest"],
                min_points=CONFIG["min_points"], value_col="val",
                station_elev_col="elev", apply_lapse=False
            )
            assert vals.size == H * W, f"IDW returned {vals.size} cells but grid is {H*W}"
            data_vars[name][ti, :, :] = vals.reshape(H, W)

# assemble dataset
ds = xr.Dataset(
    {
        **{k: xr.DataArray(v, coords=coords, dims=("time","y","x"))
           for k, v in data_vars.items()},
        "elev": xr.DataArray(
            dem1k_data.astype(np.float32),
            coords={"y": y_centers, "x": x_centers},
            dims=("y","x"),
        ),
    },
    attrs={
        "title": "Hourly predictor stacks on 1-km grid",
        "lapse_K_per_m": CONFIG["lapse_K_per_m"],
        "idw_power": CONFIG["idw_power"],
        "k_nearest": CONFIG["k_nearest"],
    }
)

# Coordinate metadata (meters)
ds["x"].attrs.update({
    "units": "m",
    "standard_name": "projection_x_coordinate",
    "long_name": "x coordinate of projection",
})
ds["y"].attrs.update({
    "units": "m",
    "standard_name": "projection_y_coordinate",
    "long_name": "y coordinate of projection",
})

# Make geospatial and CF-compliant
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds = ds.rio.write_crs(dem1k_profile["crs"], grid_mapping_name="spatial_ref")
ds = ds.rio.write_transform(dem1k_profile["transform"])

# Ensure each data variable points to the grid mapping
for v in ds.data_vars:
    ds[v].attrs["grid_mapping"] = "spatial_ref"

# Add a geotransform
A = dem1k_profile["transform"]
ds.attrs["GeoTransform"] = f"{A.c} {A.a} {A.b} {A.f} {A.d} {A.e}"

C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_27180\3943144887.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
Hourly surfaces:   0%|                                         | 0/5832 [00:00<?, ?it/s]

[2024-10-01 00:00Z] stations rows: 29 | mros rows: 0 | imerg rows: 0 | vars_ok: Ta=1, Td=1, Tw=1, RH=1, MRoS=0, PLP=0


[2024-10-01 00:00Z] Station points available: 311


[2024-10-01 00:00Z] Station points available: 311
[2024-10-01 00:00Z] Station points available: 311


Hourly surfaces:   0%|                               | 1/5832 [00:00<1:04:22,  1.51it/s]

...

[2025-05-31 22:00Z] MRoS points available: 0
[2025-05-31 22:00Z] IMERG points available: 0
[2025-05-31 23:00Z] stations rows: 34 | mros rows: 0 | imerg rows: 0 | vars_ok: Ta=1, Td=1, Tw=1, RH=1, MRoS=0, PLP=0


[2025-05-31 23:00Z] Station points available: 329
[2025-05-31 23:00Z] Station points available: 329


[2025-05-31 23:00Z] Station points available: 329
[2025-05-31 23:00Z] Station points available: 329


Hourly surfaces: 100%|████████████████████████████| 5832/5832 [1:20:04<00:00,  1.21it/s]


[2025-05-31 23:00Z] MRoS points available: 0
[2025-05-31 23:00Z] IMERG points available: 0


In [17]:
# -------------------- Save NetCDFs ------------------------------------
out_dir = Path(CONFIG["out_dir"]); out_dir.mkdir(parents=True, exist_ok=True)
out_nc = out_dir / "hourly_predictors_1km.nc"

# Ensure time is tz-naive
if hasattr(ds.indexes["time"], "tz") and ds.indexes["time"].tz is not None:
    ds = ds.assign_coords(time=ds.indexes["time"].tz_localize(None))

# Reassert spatial metadata (idempotent & safe)
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds = ds.rio.write_crs(dem1k_profile["crs"])               # <- use DEM CRS object
ds = ds.rio.write_transform(dem1k_profile["transform"])   # <- use DEM affine


# CF link each data var to the grid mapping (spatial_ref)
for v in ds.data_vars:
    ds[v].attrs.setdefault("grid_mapping", "spatial_ref")

# Build encoding per variable (match chunks to dims!)
def _chunks_for(da):
    # cap chunk sizes to something sane
    if da.dims == ("time", "y", "x"):
        return (min(24, da.sizes["time"]),
                min(256, da.sizes["y"]),
                min(256, da.sizes["x"]))
    if da.dims == ("y", "x"):
        return (min(256, da.sizes["y"]),
                min(256, da.sizes["x"]))
    return None  # scalar or unusual dims

encoding = {}
for name, da in ds.data_vars.items():
    ch = _chunks_for(da)
    encoding[name] = ({"zlib": True, "complevel": 4, "chunksizes": ch}
                      if ch is not None else {"zlib": True, "complevel": 4} if da.ndim > 0 else {})

# (coords like x/y/time generally don’t need custom encoding)
ds.to_netcdf(out_nc, engine="netcdf4", encoding=encoding)
print(f"Wrote {out_nc} using netCDF4 (compressed).")


Wrote C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\hourly_predictors_1km.nc using netCDF4 (compressed).


In [18]:
# # Check netcdf
# from netCDF4 import Dataset

# nc = Dataset(out_nc, mode="r")

# # Dimensions
# print("\nDimensions:")
# for name, dim in nc.dimensions.items():
#     print(f"  {name}: {len(dim)}")

# # Variables
# print("\nVariables:")
# for name, var in nc.variables.items():
#     print(f"  {name}: shape={var.shape}, dtype={var.dtype}, attrs={ {k: v for k, v in var.__dict__.items()} }")

# # Check the data
# out_nc = Path(CONFIG["out_dir"]) / "hourly_predictors_1km.nc"
# ds = xr.open_dataset(out_nc)

# # Print a quick summary again
# print(ds)

# # Inspect first few timesteps for one variable (e.g. temp_air)
# print("\nFirst 2 timesteps of temp_air, temp_wet, temp_dew, rh, mros_plp_proxy, plp: at 5x5 corner:")
# print(ds["temp_air"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["temp_wet"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["temp_dew"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["rh"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["mros_plp_proxy"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)
# print(ds["plp"].isel(time=slice(0,10), y=slice(0,5), x=slice(0,5)).values)

# ds.close()


In [19]:
# -------------------- Quick Plotting ------------------------------------
from pyproj import CRS

def quicklook_hour(
    ds, t, st_t, mros_t, out_png,
    vars_to_show=("plp","mros_plp_proxy","temp_air","temp_dew","temp_wet","rh")
):
    # match time index
    times_ds = pd.to_datetime(ds.time.values).floor("h")
    t_floor  = pd.to_datetime(t).floor("h")
    if t_floor not in times_ds.values:
        print(f"No matching time {t_floor} in dataset for quicklook.")
        return
    ti = int(np.where(times_ds == t_floor)[0][0])

    # axes extent (xmin, xmax, ymin, ymax)
    xvals = ds["x"].values
    yvals = ds["y"].values
    xmin, xmax = float(np.min(xvals)), float(np.max(xvals))
    ymin, ymax = float(np.min(yvals)), float(np.max(yvals))
    extent = [xmin, xmax, ymin, ymax]

    keep = [v for v in vars_to_show if v in ds.data_vars]
    if not keep:
        print("No matching variables to plot.")
        return
    ncols, nrows = 3, int(np.ceil(len(keep)/3))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows), squeeze=False)
    fig.suptitle(f"Quicklook @ {t_floor:%Y-%m-%d %H:%MZ}", fontsize=14)

    # dataset CRS (fallback to configured)
    target_crs = ds.rio.crs or CRS.from_user_input(CONFIG["proj_fallback"])
    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

    # --- project & CLIP stations ---
    st_x = np.empty(0)
    st_y = np.empty(0)
    if len(st_t):
        sx, sy = tf.transform(st_t["lon"].values, st_t["lat"].values)
        sx = np.asarray(sx); sy = np.asarray(sy)
        smask = (sx >= xmin) & (sx <= xmax) & (sy >= ymin) & (sy <= ymax) & np.isfinite(sx) & np.isfinite(sy)
        st_x, st_y = sx[smask], sy[smask]

    # --- project & CLIP MRoS ---
    mo_x = np.empty(0)
    mo_y = np.empty(0)
    if len(mros_t):
        mx, my = tf.transform(mros_t["lon"].values, mros_t["lat"].values)
        mx = np.asarray(mx); my = np.asarray(my)
        mmask = (mx >= xmin) & (mx <= xmax) & (my >= ymin) & (my <= ymax) & np.isfinite(mx) & np.isfinite(my)
        mo_x, mo_y = mx[mmask], my[mmask]

    for i, var in enumerate(keep):
        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        # color scaling
        if var in ("plp", "mros_plp_proxy", "rh"):
            im = ax.imshow(arr, origin="lower", extent=extent, aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="lower", extent=extent, aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("x"); ax.set_ylabel("y")

        # overlay (after clipping)
        if st_x.size:
            ax.scatter(st_x, st_y, s=15, c="white", edgecolor="k",
                    marker="o", linewidths=0.5, label="Stations", zorder=3)
        if mo_x.size:
            ax.scatter(mo_x, mo_y, s=25, c="red", edgecolor="k",
                    marker="^", linewidths=0.6, label="MRoS", zorder=3)

        ax.legend(loc="upper right", frameon=True, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # turn off any leftover panels
    for j in range(len(keep), nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(
        f"Saved quicklook: {out_png} | plotted {st_x.size} stations, {mo_x.size} MRoS (clipped to DEM)"
    )


# -------------------- Loop --------------------
quick_dir = Path(CONFIG["out_dir"]) / "maps"
quick_dir.mkdir(parents=True, exist_ok=True)

# day = "2025-03-04"
# all_times = pd.to_datetime(ds.time.values).floor("h")  # dataset times
# mask = all_times.normalize() == pd.to_datetime(day)
# sample_hours = all_times[mask]
sample_hours = pd.to_datetime(ds.time.values)[::max(1, len(ds.time)//20)]
# print(f"Found {len(sample_hours)} timesteps on {day}")

for t in sample_hours:
    t_floor = pd.to_datetime(t).floor("h")  # tz-naive

    # Ensure obs times are made tz-naive before comparison
    st_t   = st_hr[st_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]
    mros_t = mros[mros["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]

    print(f"[{t_floor}] Stations: {len(st_t)}, MRoS: {len(mros_t)}")

    quicklook_hour(ds, t_floor, st_t, mros_t,
                   out_png=quick_dir / f"quick_{print_time(t_floor).replace(':','-')}.png")


[2024-10-01 00:00:00] Stations: 311, MRoS: 0
Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\quick_2024-10-01 00-00Z.png | plotted 311 stations, 0 MRoS (clipped to DEM)
[2024-10-13 03:00:00] Stations: 349, MRoS: 0
Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\quick_2024-10-13 03-00Z.png | plotted 349 stations, 0 MRoS (clipped to DEM)
[2024-10-25 06:00:00] Stations: 348, MRoS: 0
Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\quick_2024-10-25 06-00Z.png | plotted 348 stations, 0 MRoS (clipped to DEM)
[2024-11-06 09:00:00] Stations: 58, MRoS: 0
Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\quick_2024-11-06 09-00Z.png | plotted 58 stations, 0 MRoS (clipped to DEM)
[2024-11-1